# Extension 2 — LLM with Tool Calling

**Thesis Section: Chapter 6.2 — Tool-augmented LLM**

Tests GPT-4o with access to Python statistical analysis tools.
Instead of receiving pre-computed statistics in the prompt text, the
model actively calls tools to investigate the signal before classifying.

**Available tools**:
- `get_statistics` — global mean/std/min/max/percentiles
- `detect_spikes` — 4σ local spike detection
- `detect_drift` — rolling-mean range test
- `detect_frozen` — rolling-std frozen test
- `get_segment` — retrieve a specific index range

**Comparison**: Tool-calling vs Approach 3 text-stats (same statistical
information, different delivery mechanism).

**Hypothesis**: Active tool use lets the model focus investigation on
relevant statistics, improving accuracy on ambiguous signals.

**Requires**: `OPENAI_API_KEY` in `.env`

In [ ]:
import sys
sys.path.insert(0, '../..')
from dotenv import load_dotenv
load_dotenv('../../.env')

import os
import numpy as np

from src.data.timeseer_client import fetch_series_api, list_series_api
from src.data.ground_truth import GROUND_TRUTH
from src.prompts.templates import MCQ_CATEGORIES
from src.inference.tool_calling import ask_openai_with_tools
from src.parsing.extract import extract_category
from src.evaluation.metrics import compute_metrics
from src.evaluation.report import print_summary_table, save_results

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env'
AREA = 'Reactor 1'
print('Imports OK')

In [ ]:
tags = list_series_api(AREA)
print(f'{len(tags)} signals in {AREA}')

In [ ]:
print('Running TOOL-CALLING baseline...')
results = []
tool_call_counts = []

for tag in tags:
    vals, idx = fetch_series_api(tag, AREA)
    if vals is None:
        continue

    answer, tool_log = ask_openai_with_tools(vals, tag, MCQ_CATEGORIES)
    cat, lbl = extract_category(answer)
    gt = GROUND_TRUTH.get(tag, '?')

    n_calls = len(tool_log)
    tool_call_counts.append(n_calls)
    results.append({'Tag': tag, 'gt': gt, 'Category': cat,
                    'Label': lbl, 'ToolCalls': n_calls})
    print(f'  {tag:<30} → {cat}) {lbl}  (GT: {gt})  [{n_calls} tool calls]')

    if tool_log:
        print('    Tool trace:')
        for t in tool_log:
            print(f'      {t}')

In [ ]:
print_summary_table(results, title=f'TOOL CALLING — {AREA}', model_name='GPT-4o')

evaluated = [r for r in results if r['gt'] != '?']
preds = [r['Category'] for r in evaluated]
gts   = [r['gt']       for r in evaluated]
m = compute_metrics(preds, gts)

import numpy as np
print(f'\nAccuracy : {m["accuracy"]:.3f}')
print(f'Macro F1 : {m["f1"]:.3f}')
print(f'Avg tool calls per signal: {np.mean(tool_call_counts):.1f}')
print(f'Max tool calls: {max(tool_call_counts)}')

save_results(results, f'../../data/gpt4o_tools_{AREA.replace(" ","_")}.txt',
             header=f'GPT-4o Tool Calling | {AREA}')

In [ ]:
# Which tool did the model call most often?
from collections import Counter
import matplotlib.pyplot as plt

# Re-run a single example with verbose tool trace for illustration
DEMO_TAG = 'R1-AT-101-PH'
vals, idx = fetch_series_api(DEMO_TAG, AREA)
answer, tool_log = ask_openai_with_tools(vals, DEMO_TAG, MCQ_CATEGORIES)

print(f'\nDemo: {DEMO_TAG}')
print(f'Final answer: {answer[:200]}')
print(f'\nTool calls made ({len(tool_log)}):')
for t in tool_log:
    print(f'  {t}')

# Tool call frequency chart across all signals
all_tools_used = []
for r in results:
    # tool_log is not stored per-result above, but ToolCalls count is
    pass  # Extend by re-running with tool_log storage if needed
print('\nNote: re-run with tool_log storage to get per-tool frequency chart.')